# Pre-Flight Invariant AST Security Gating for AutoGen Code Execution

This cookbook demonstrates how to integrate `btp-guard` (Bartholomew Trust Protocol) into AutoGen's `LocalCommandLineCodeExecutor` to provide deterministic, in-memory pre-flight AST gating for code blocks before dispatch.

### Key Features
* **Sub-50µs Overhead:** In-memory inspection for Python, Bash, and SQL code.
* **Proactive Interception:** Pre-emptively traps high-risk operations (e.g., `rm -rf`, `DROP TABLE`, credential exposures) before host execution.

In [ ]:
%pip install --quiet autogen btp-guard

## Step 1: Extend `LocalCommandLineCodeExecutor`
Intercept incoming code blocks and evaluate them against BTP security invariants prior to execution.

In [ ]:
from autogen.coding import LocalCommandLineCodeExecutor
from btp_guard import guard

class SecureCodeExecutor(LocalCommandLineCodeExecutor):
    def execute_code_blocks(self, code_blocks):
        for block in code_blocks:
            # Sub-50µs in-memory evaluation
            is_safe, reason = guard.evaluate(block.language, block.code)
            if not is_safe:
                return 1, f"[BTP Security Veto] Execution aborted: {reason}", None
                
        return super().execute_code_blocks(code_blocks)

## Step 2: Validating Allowed Execution
Standard operations execute without noticeable overhead.

In [ ]:
class CodeBlock:
    def __init__(self, language, code):
        self.language = language
        self.code = code

executor = SecureCodeExecutor(work_dir="coding")

safe_block = CodeBlock("python", "print('AutoGen execution secure.')")
exit_code, logs, _ = executor.execute_code_blocks([safe_block])

print(f"Exit Code: {exit_code}")
print(f"Logs: {logs}")

## Step 3: Intercepting Security Violations
Destructive commands (such as `rm -rf /` or DDL purges) trigger an immediate BTP veto before hitting the host/shell.

In [ ]:
unsafe_block = CodeBlock("bash", "rm -rf /")
exit_code, logs, _ = executor.execute_code_blocks([unsafe_block])

print(f"Exit Code: {exit_code}")
print(f"Logs: {logs}")

## Security Considerations & Limitations

> **Important:** Static AST inspection serves as a zero-latency heuristic pre-filter for obvious high-risk patterns. It does not replace container or OS isolation (e.g., Docker or sandboxes). Dynamic constructs like `eval()` or dynamic string concatenation sit outside static syntax tree boundaries. Always pair pre-flight gating with robust runtime sandboxing for defense-in-depth.